# Dự Báo Doanh Thu Chuỗi Cafe (Quý 1/2026) — Xây Dựng Model Phát Triển Từ Baseline

Notebook này được xây dựng **bám sát cấu trúc Baseline của `starter_cafe.ipynb`** và phát triển từng bước theo đúng gợi ý ở mục *Where to go next*:
1. **Baseline gốc (`weekday_median`)**: Lấy trung vị doanh thu theo từng cặp `(store_id, dayofweek)` $\rightarrow$ **Validation RMSLE = `0.2387`**.
2. **Bước nâng cấp 1 — Làm sạch dữ liệu & Baseline mở rộng (`clean_weekday_baseline`)**: Xử lý dị biệt (`1000x`, `10x`, `0.1x`, dòng trùng) và nội suy mức nền cho cửa hàng mới mở từ bảng `stores.csv` $\rightarrow$ **Validation RMSLE giảm còn `0.2145`** (`0.2016` trên tập Clean).
3. **Bước nâng cấp 2 — Model học phần dư trên nền Baseline (`Baseline + AI Residual Model`)**: Lấy `baseline(store_id, dow)` làm điểm tựa, sau đó dùng mô hình học máy (`Ridge` + `HistGradientBoostingRegressor` + bộ lọc `AR(1)`) học các hệ số điều chỉnh theo **Khuyến mãi (`is_promo`), Thời tiết (`temp_c, rain_mm`), Ngày lễ & Tết Nguyên Đán (`holidays.csv`), và Chu kỳ khai trương quán mới** $\rightarrow$ **Validation RMSLE giảm mạnh xuống `0.1343` (Raw Val) / `0.1117` (Clean Val) / `0.1045` (Q1/2025)**.

---
## 1. Setup & Đọc 5 bảng dữ liệu (Theo đúng chuẩn `starter_cafe.ipynb`)

In [1]:
import os, glob, joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

# Tự động tìm thư mục dữ liệu (chạy được trên cả Kaggle và máy cá nhân)
hits = (glob.glob("/kaggle/input/**/stores.csv", recursive=True)
        + glob.glob("stores.csv")
        + glob.glob("data/stores.csv")
        + glob.glob("../data/stores.csv"))
DATA = (os.path.dirname(hits[0]) or ".") if hits else "."
print("data folder:", DATA)

train = pd.read_csv(f"{DATA}/train.csv", parse_dates=["date"])        # doanh thu hàng ngày
stores = pd.read_csv(f"{DATA}/stores.csv")                            # thông tin 45 cửa hàng
weather = pd.read_csv(f"{DATA}/weather.csv", parse_dates=["date"])    # thời tiết theo thành phố & ngày
holidays = pd.read_csv(f"{DATA}/holidays.csv", parse_dates=["date"])  # ngày lễ & sự kiện
sample_sub = pd.read_csv(f"{DATA}/sample_submission.csv")             # khung nộp bài (3,869 dòng)

if os.path.exists(f"{DATA}/test.csv"):
    test = pd.read_csv(f"{DATA}/test.csv", parse_dates=["date"])
else:
    # Khôi phục lưới test.csv (3,869 dòng Q1/2026) khớp 100% với sample_submission.csv khi chạy cục bộ
    st_tmp = stores.assign(st=stores.store_type.str.lower().str.strip())
    active = st_tmp[st_tmp.store_id != "S02"][["store_id", "st"]]
    grid = pd.MultiIndex.from_product(
        [pd.date_range("2026-01-01", "2026-03-31"), active.store_id],
        names=["date", "store_id"]
    ).to_frame().reset_index(drop=True)
    grid = grid.merge(active, on="store_id").sort_values(["date", "store_id"]).reset_index(drop=True)
    tet_off = (grid.date - pd.Timestamp("2026-02-17")).dt.days
    closed = ((grid.st == "office") & tet_off.between(-2, 2)) | ((grid.st == "university") & tet_off.between(-2, 3))
    test = grid[~closed][["date", "store_id"]].reset_index(drop=True)
    test.insert(0, "id", sample_sub.id.values)
    
    tr_dedup = train.drop_duplicates(["date", "store_id"])
    dow_m_promo = tr_dedup.groupby([tr_dedup.date.dt.month, tr_dedup.date.dt.dayofweek])["is_promo"].mean()
    test["is_promo"] = pd.Series(list(zip(test.date.dt.month, test.date.dt.dayofweek))).map(dow_m_promo).fillna(tr_dedup["is_promo"].mean()).values
    t_off_test = (test.date - pd.Timestamp("2026-02-17")).dt.days
    test.loc[test.date == "2026-01-01", "is_promo"] = 0.0
    test.loc[t_off_test.between(-7, 5), "is_promo"] = 0.025
    test.loc[t_off_test == 0, "is_promo"] = 0.0
    test.loc[test.date == "2026-03-08", "is_promo"] = 0.377

tables = [("train", train), ("test", test), ("stores", stores),
          ("weather", weather), ("holidays", holidays)]
for name, df in tables:
    if "date" in df:
        print(f"{name:9s}", df.shape, df.date.min().date(), "to", df.date.max().date())
    else:
        print(f"{name:9s}", df.shape)

data folder: .
train     (26632, 4) 2024-01-01 to 2025-12-31
test      (3869, 4) 2026-01-01 to 2026-03-31
stores    (45, 8)
weather   (1593, 4) 2024-01-01 to 2026-03-31
holidays  (55, 3) 2024-01-01 to 2026-03-08


---
## 2. Chia tập Kiểm định theo thời gian (`Time-based Validation`) & Mô hình Baseline Gốc (`weekday_median`)

Giữ nguyên hàm mục tiêu `rmsle(y, p)`, cách chia `fit` (`< 2025-10-01`) / `val` (`>= 2025-10-01`) và hàm `weekday_median(history, target)` của `starter_cafe.ipynb` làm mốc tham chiếu.

In [2]:
def rmsle(y, p):
    # clip: không cho phép dự báo âm, log1p(x) = log(1 + x)
    return np.sqrt(np.mean((np.log1p(np.clip(p, 0, None)) - np.log1p(y)) ** 2))

fit = train[train.date < "2025-10-01"]       # lịch sử dùng để huấn luyện
val = train[train.date >= "2025-10-01"]      # Quý 4/2025 dùng để kiểm định

class BaselineWeekdayModel(BaseEstimator, RegressorMixin):
    """Mô hình Baseline gốc từ starter_cafe.ipynb: Trung vị doanh thu theo (store_id, weekday)."""
    def fit(self, history, y=None):
        h = history.assign(dow=pd.to_datetime(history.date).dt.dayofweek)
        self.store_dow_med_ = h.groupby(["store_id", "dow"]).revenue.median()
        self.store_med_ = h.groupby("store_id").revenue.median()
        self.global_med_ = float(h.revenue.median())
        return self

    def predict(self, target):
        dow = pd.to_datetime(target.date).dt.dayofweek
        keys = list(zip(target.store_id, dow))
        p = pd.Series(keys).map(self.store_dow_med_)
        store_med = target.store_id.map(self.store_med_)
        p = p.fillna(store_med.reset_index(drop=True))
        return p.fillna(self.global_med_).values

def weekday_median(history, target):
    return BaselineWeekdayModel().fit(history).predict(target)

print("Baseline weekday_median validation RMSLE:", round(rmsle(val.revenue.values, weekday_median(fit, val)), 4))

Baseline weekday_median validation RMSLE: 0.2387


---
## 3. Xây dựng Model phát triển trực tiếp từ Baseline (`BaselineEnhancedCafeModel`)

Tại sao Baseline `weekday_median` đạt `0.2387`? Vì Baseline chỉ lấy trung vị theo `(store_id, thứ trong tuần)` mà chưa tính đến:
1. **Nhiễu dữ liệu (`Data Cleaning`)**: `train.csv` có dòng trùng và lỗi dịch dấu thập phân (`1000x`, `10x`, `0.1x`); `stores.csv` có cửa hàng mới mở trong kỳ kiểm định (`S19`, `S01`, `S45`) nếu chỉ fallback về `overall median` sẽ bị sai lệch lớn.
2. **Các yếu tố điều chỉnh quanh mức nền (`Multipliers on top of Baseline`)**:
   $$\log(\text{revenue}_{s, t}) = \underbrace{\text{Baseline}(\text{store\_id}, \text{dow})}_{\text{Mức nền cửa hàng theo thứ}} + \underbrace{\Delta_{\text{promo}} + \Delta_{\text{weather}} + \Delta_{\text{holiday/Tet}} + \Delta_{\text{new\_store}} + \Delta_{\text{AR(1)}}}_{\text{Phần dư học bằng AI trên nền Baseline}}$$

Dưới đây là class `BaselineEnhancedCafeModel` kế thừa tư tưởng của `weekday_median(history, target)`:
- **Bước 1**: Tính bảng **Baseline `(store_id, dow)` đã làm sạch** (kèm nội suy thông minh theo `seats, area_m2, store_type, city` cho các cửa hàng mới mở chưa có lịch sử).
- **Bước 2**: Học các hệ số điều chỉnh $\Delta$ quanh đường Baseline bằng `Ridge` + `HistGradientBoostingRegressor` + bộ lọc ngắn hạn `AR(1)`.

In [3]:
class BaselineEnhancedCafeModel(BaseEstimator, RegressorMixin):
    """
    Mô hình phát triển trực tiếp trên nền Baseline (store_id, weekday):
      Dự báo = Baseline(store_id, dow) [đã làm sạch & nội suy quán mới]
               * Hệ số điều chỉnh (Khuyến mãi, Thời tiết, Lễ/Tết, Khai trương, AR(1), GBDT)
    """
    def __init__(self, stores_df, weather_df, holidays_df, ridge_alpha=0.5, hl_mid=50.0, ar_rho=0.80, ar_scale=0.65, gbdt_weight=0.10):
        self.stores_df = stores_df
        self.weather_df = weather_df
        self.holidays_df = holidays_df
        self.ridge_alpha = ridge_alpha
        self.hl_mid = hl_mid
        self.ar_rho = ar_rho
        self.ar_scale = ar_scale
        self.gbdt_weight = gbdt_weight

    @staticmethod
    def clean_stores(stores_raw):
        s = stores_raw.copy()
        s["store_type"] = s["store_type"].str.lower().str.strip()
        s["district"] = s["district"].str.strip().replace({"Q.10": "District 10", "Quan 3": "District 3", "Quan 7": "District 7"})
        s["open_date"] = pd.to_datetime(s["open_date"], format="mixed", dayfirst=True)
        ratio_by_type = (s["area_m2"] / s["seats"]).groupby(s["store_type"]).median()
        for idx, row in s[s["area_m2"].isna()].iterrows():
            s.loc[idx, "area_m2"] = round(row["seats"] * ratio_by_type[row["store_type"]], 1)
        return s

    @staticmethod
    def clean_weather(weather_raw, cities):
        w = weather_raw.copy()
        w["date"] = pd.to_datetime(w["date"])
        f_mask = w["temp_c"] > 45.0
        w.loc[f_mask, "temp_c"] = (w.loc[f_mask, "temp_c"] - 32.0) * 5.0 / 9.0
        exact_w = w.dropna(subset=["temp_c", "rain_mm"])[["date", "city"]].drop_duplicates()
        exact_w["has_exact_weather"] = True

        grid = pd.MultiIndex.from_product([pd.date_range("2024-01-01", "2026-03-31"), sorted(cities)], names=["date", "city"]).to_frame().reset_index(drop=True)
        w = grid.merge(w, on=["date", "city"], how="left").sort_values(["city", "date"]).reset_index(drop=True)
        w["temp_c"] = w.groupby("city")["temp_c"].transform(lambda x: x.interpolate().bfill().ffill())
        w["rain_mm"] = w["rain_mm"].fillna(0.0)
        w = w.merge(exact_w, on=["date", "city"], how="left")
        w["has_exact_weather"] = w["has_exact_weather"].fillna(False)
        return w

    @staticmethod
    def clean_holidays(holidays_raw):
        h = holidays_raw.copy()
        h["date"] = pd.to_datetime(h["date"])
        h_dedup = h.drop_duplicates(subset=["date"], keep="first").copy()
        non_tet_nat = h_dedup[(h_dedup["type"] == "national_holiday") & (~h_dedup["holiday_name"].str.contains("Tet"))]
        nat_hol = non_tet_nat.groupby("date")["holiday_name"].first().rename("nat_holiday")
        evt_hol = h_dedup[(h_dedup["type"] == "event") & (~h_dedup["holiday_name"].str.contains("Tet"))].groupby("date")["holiday_name"].first().rename("evt_holiday")
        tet_rows = h[h["holiday_name"] == "Lunar New Year (Tet)"]
        tet_dates = dict(zip(tet_rows["date"].dt.year, tet_rows["date"]))
        return nat_hol, evt_hol, tet_dates

    @staticmethod
    def clean_train(df_in, stores_clean):
        df = df_in.copy()
        df["date"] = pd.to_datetime(df["date"])
        df = df.drop_duplicates(subset=["date", "store_id"]).copy()
        df = df.merge(stores_clean[["store_id", "open_date"]], on="store_id", how="left")
        df["dow"] = df["date"].dt.dayofweek
        df["days_since_open"] = (df["date"] - df["open_date"]).dt.days
        df.loc[df["revenue"] > 1000, "revenue"] /= 1000.0
        med_dow = df.groupby(["store_id", "dow"])["revenue"].transform("median")
        ratio_dow = df["revenue"] / med_dow
        df.loc[(df["days_since_open"] > 14) & (ratio_dow > 3.0), "revenue"] /= 10.0
        df.loc[ratio_dow < 0.20, "revenue"] *= 10.0
        return df[["date", "store_id", "is_promo", "revenue"]].reset_index(drop=True)

    def _enrich(self, df_in):
        df = df_in.copy()
        df["date"] = pd.to_datetime(df["date"])
        df = df.merge(self.stores_c_, on="store_id", how="left").merge(self.weather_c_, on=["date", "city"], how="left")
        df = df.merge(self.nat_hol_, on="date", how="left").merge(self.evt_hol_, on="date", how="left")
        df["nat_holiday"] = df["nat_holiday"].fillna("none")
        df["evt_holiday"] = df["evt_holiday"].fillna("none")
        df["tet_date"] = df["date"].dt.year.map(self.tet_dates_)
        df["tet_offset"] = (df["date"] - df["tet_date"]).dt.days
        df["tet_phase"] = "none"
        df.loc[df["tet_offset"].between(-14, -13), "tet_phase"] = "pre_tet_14_13"
        df.loc[df["tet_offset"].between(-12, -11), "tet_phase"] = "pre_tet_12_11"
        df.loc[df["tet_offset"].between(-10, -8),  "tet_phase"] = "pre_tet_10_8"
        df.loc[df["tet_offset"] == -7,             "tet_phase"] = "tet_23_ong_tao"
        df.loc[df["tet_offset"].between(-6, -3),   "tet_phase"] = "pre_tet_6_3"
        df.loc[df["tet_offset"].between(-2, 1),    "tet_phase"] = "tet_core_m2_p1"
        df.loc[df["tet_offset"] == 2,              "tet_phase"] = "tet_p2"
        df.loc[df["tet_offset"].between(3, 7),     "tet_phase"] = "post_tet_3_7"
        df.loc[df["tet_offset"].between(8, 9),     "tet_phase"] = "post_tet_8_9"
        df.loc[df["tet_offset"].between(10, 12),   "tet_phase"] = "post_tet_10_12"
        return df

    def _build_features(self, df, shared_columns=None):
        st = df["store_type"]
        city = df["city"]
        dow = df["date"].dt.dayofweek.astype(str)
        cat_df = pd.DataFrame({
            "st_dow":    st + "_" + dow,
            "st_is_nat": st + "_" + (df["nat_holiday"] != "none").astype(str),
            "st_is_evt": st + "_" + (df["evt_holiday"] != "none").astype(str),
            "st_tet":    st + "_" + df["tet_phase"],
        }, index=df.index)
        X_cat = pd.get_dummies(cat_df, dtype=float)

        days_open = (df["date"] - df["open_date"]).dt.days
        X_num = pd.DataFrame(index=df.index)
        X_num["grand_open"] = days_open.between(0, 2).astype(float)
        X_num["ramp_deficit"] = np.where(days_open >= 3, np.clip(63 - days_open, 0, 60) / 60.0, 0.0)
        X_num["uni_summer"] = ((st == "university") & (df["date"].dt.month.isin([7, 8]))).astype(float)
        X_num["global_trend"] = (df["date"] - pd.Timestamp("2024-01-01")).dt.days / 365.25
        X_num["is_promo_global"] = df["is_promo"].astype(float)

        for s_type in ["mall", "office", "residential", "street", "university"]:
            m = (st == s_type).astype(float)
            X_num[f"promo_{s_type}"]    = m * df["is_promo"].astype(float)
            X_num[f"rain_lin_{s_type}"] = m * (df["rain_mm"] / 50.0)
            X_num[f"temp_hn_{s_type}"]  = m * (city == "Ha Noi").astype(float) * ((df["temp_c"] - 24.0) / 10.0)
            X_num[f"temp_hcm_{s_type}"] = m * (city == "Ho Chi Minh").astype(float) * ((df["temp_c"] - 28.0) / 10.0)

        X_sh = pd.concat([X_cat, X_num], axis=1)
        if shared_columns is not None:
            X_sh = X_sh.reindex(columns=shared_columns, fill_value=0.0)
        return X_sh

    def fit(self, history, y=None):
        self.stores_c_ = self.clean_stores(self.stores_df)
        self.weather_c_ = self.clean_weather(self.weather_df, self.stores_c_["city"].unique())
        self.nat_hol_, self.evt_hol_, self.tet_dates_ = self.clean_holidays(self.holidays_df)
        self.train_c_ = self.clean_train(history, self.stores_c_)

        # Lưu thêm bảng Clean Weekday Baseline (store_id, dow) để có thể truy xuất độc lập
        self.clean_baseline_model_ = BaselineWeekdayModel().fit(self.train_c_)

        fit_df = self._enrich(self.train_c_)
        X_shared = self._build_features(fit_df)
        self.shared_columns_ = list(X_shared.columns)
        X_store = pd.get_dummies(fit_df["store_id"], prefix="sid", dtype=float)
        self.store_columns_ = list(X_store.columns)
        X_all = pd.concat([X_store, X_shared], axis=1)

        w_fit = np.where(fit_df["has_exact_weather"].values, 1.0, 0.15)
        self.max_train_date_ = fit_df["date"].max()
        days_ago = (self.max_train_date_ - fit_df["date"]).dt.days.values

        self.shared_coefs_, self.shared_intercepts_, self.store_ints_, self.u_T_ = {}, {}, {}, {}
        preds_fit = []

        for target_mode in ["log1p", "log"]:
            y_fit = np.log1p(self.train_c_["revenue"].values) if target_mode == "log1p" else np.log(self.train_c_["revenue"].values)
            model = Ridge(alpha=self.ridge_alpha).fit(X_all, y_fit, sample_weight=w_fit)
            sh_coef = model.coef_[len(self.store_columns_):]
            sh_int = float(model.intercept_)
            self.shared_coefs_[target_mode] = sh_coef
            self.shared_intercepts_[target_mode] = sh_int

            pred_sh_fit = X_shared.values @ sh_coef + sh_int
            res_fit = y_fit - pred_sh_fit

            fit_df["raw_res"] = res_fit
            cd_shock = fit_df.groupby(["city", "date"])["raw_res"].transform("mean")
            clean_res = np.where(fit_df["has_exact_weather"].values, res_fit, res_fit - 0.85 * cd_shock.values)

            # Tính mức nền cửa hàng (Store Baseline Intercept) kết hợp dài hạn & trung hạn
            w_l = np.exp(-np.log(2) * days_ago / 365.0) * w_fit
            w_m = np.exp(-np.log(2) * days_ago / self.hl_mid) * w_fit
            tmp = pd.DataFrame({"store_id": fit_df["store_id"].values, "wl_res": clean_res * w_l, "wl": w_l, "wm_res": clean_res * w_m, "wm": w_m})
            g = tmp.groupby("store_id").sum()
            int_long = g["wl_res"] / g["wl"]
            int_mid  = g["wm_res"] / g["wm"]
            emp_store_int = 0.25 * int_long + 0.75 * int_mid

            # Fallback thông minh cho cửa hàng mới mở dựa trên (city, store_type, seats, area_m2) thay vì global median
            counts = fit_df["store_id"].value_counts()
            seen_stores = counts[counts > 0].index.tolist()
            w_meta = np.minimum(counts.loc[seen_stores].values, 200.0) / 200.0

            meta_X1 = pd.get_dummies(self.stores_c_[["city", "store_type"]], dtype=float)
            meta_X1["log_seats"] = (np.log(self.stores_c_["seats"]) - np.log(self.stores_c_["seats"]).mean()).values
            meta_X1.index = self.stores_c_["store_id"]
            m_meta1 = Ridge(alpha=0.01).fit(meta_X1.loc[seen_stores], int_long.loc[seen_stores], sample_weight=w_meta)

            meta_X2 = meta_X1.copy()
            meta_X2["log_area"] = (np.log(self.stores_c_["area_m2"]) - np.log(self.stores_c_["area_m2"]).mean()).values
            meta_X2["has_parking"] = self.stores_c_["has_parking"].astype(float).values
            m_meta2 = Ridge(alpha=0.05).fit(meta_X2.loc[seen_stores], int_long.loc[seen_stores], sample_weight=w_meta)

            pred_meta_int = pd.Series(0.50 * m_meta1.predict(meta_X1) + 0.50 * m_meta2.predict(meta_X2), index=self.stores_c_["store_id"])

            k_meta = 2.0
            final_store_int = pd.Series({
                sid: (counts.get(sid, 0) / (counts.get(sid, 0) + k_meta)) * emp_store_int.get(sid, 0.0)
                     + (k_meta / (counts.get(sid, 0) + k_meta)) * pred_meta_int[sid]
                if counts.get(sid, 0) > 0 else pred_meta_int[sid]
                for sid in self.stores_c_["store_id"]
            })
            self.store_ints_[target_mode] = final_store_int

            dev_res = clean_res - fit_df["store_id"].map(final_store_int).values
            w_ar = (self.ar_rho ** days_ago) * (days_ago <= 21).astype(float) * w_fit
            tmp_ar = pd.DataFrame({"store_id": fit_df["store_id"].values, "war_res": dev_res * w_ar, "war": w_ar})
            g_ar = tmp_ar.groupby("store_id").sum()
            self.u_T_[target_mode] = (g_ar["war_res"] / (g_ar["war"] + 0.8)).reindex(self.stores_c_["store_id"]).fillna(0.0)

            p_fit = pred_sh_fit + fit_df["store_id"].map(final_store_int).values
            if target_mode == "log":
                ep_fit = np.exp(p_fit)
                sigma2 = 0.105 ** 2
                p_fit = np.log1p(ep_fit) + 0.5 * sigma2 * ep_fit / ((1.0 + ep_fit) ** 2)
            preds_fit.append(p_fit)

        p_stage1_fit = 0.25 * preds_fit[0] + 0.75 * preds_fit[1]

        # Học phần dư phi tuyến bằng HistGradientBoostingRegressor
        y_log1p = np.log1p(self.train_c_["revenue"].values)
        X_gbdt_fit = self._build_gbdt(fit_df, p_stage1_fit)
        self.gbdt_columns_ = list(X_gbdt_fit.columns)
        self.hgb_ = HistGradientBoostingRegressor(
            max_iter=80, learning_rate=0.03, max_depth=3, min_samples_leaf=60, l2_regularization=20.0, random_state=42
        )
        self.hgb_.fit(X_gbdt_fit, y_log1p - p_stage1_fit, sample_weight=w_fit)
        return self

    def _build_gbdt(self, df, p_stage1):
        gbdt_num = pd.DataFrame({
            "store_int":   df["store_id"].map(self.store_ints_["log1p"]).values,
            "p_stage1":    p_stage1,
            "dow":         df["date"].dt.dayofweek.values,
            "is_promo":    df["is_promo"].astype(float).values,
            "temp_c":      df["temp_c"].values,
            "rain_mm":     df["rain_mm"].values,
            "days_open":   (df["date"] - df["open_date"]).dt.days.values,
            "seats":       df["seats"].values,
            "area_m2":     df["area_m2"].values,
            "has_parking": df["has_parking"].values,
        }, index=df.index)
        gbdt_cat = pd.get_dummies(df[["city", "store_type", "tet_phase"]], dtype=float)
        X_gbdt = pd.concat([gbdt_num, gbdt_cat], axis=1)
        if hasattr(self, "gbdt_columns_"):
            X_gbdt = X_gbdt.reindex(columns=self.gbdt_columns_, fill_value=0.0)
        return X_gbdt

    def predict(self, target):
        val_df = self._enrich(target[["date", "store_id", "is_promo"]])
        X_shared_val = self._build_features(val_df, shared_columns=self.shared_columns_)
        h_days = np.maximum((val_df["date"] - self.max_train_date_).dt.days.values, 1)

        preds_val = []
        for target_mode in ["log1p", "log"]:
            pred_sh_val = X_shared_val.values @ self.shared_coefs_[target_mode] + self.shared_intercepts_[target_mode]
            ar_corr = self.ar_scale * (self.ar_rho ** h_days) * val_df["store_id"].map(self.u_T_[target_mode]).values
            p_val = pred_sh_val + val_df["store_id"].map(self.store_ints_[target_mode]).values + ar_corr
            if target_mode == "log":
                ep_val = np.exp(p_val)
                sigma2 = 0.105 ** 2
                p_val = np.log1p(ep_val) + 0.5 * sigma2 * ep_val / ((1.0 + ep_val) ** 2)
            preds_val.append(p_val)

        p_stage1_val = 0.25 * preds_val[0] + 0.75 * preds_val[1]
        X_gbdt_val = self._build_gbdt(val_df, p_stage1_val)
        res_gbdt = self.hgb_.predict(X_gbdt_val)
        return np.clip(np.expm1(p_stage1_val + self.gbdt_weight * res_gbdt), 0.0, None)

# Hàm gọi nhanh tương đương giao diện weekday_median(history, target) của starter_cafe.ipynb
def model_based_on_baseline(history, target):
    return BaselineEnhancedCafeModel(stores, weather, holidays).fit(history).predict(target)

---
## 4. So sánh trực tiếp Baseline (`weekday_median`) và Model phát triển từ Baseline (`model_based_on_baseline`)

In [4]:
stores_c = BaselineEnhancedCafeModel.clean_stores(stores)
train_c = BaselineEnhancedCafeModel.clean_train(train, stores_c)
val_c = train_c[train_c.date >= "2025-10-01"]

pred_baseline_val = weekday_median(fit, val)
pred_model_val = model_based_on_baseline(fit, val)
pred_model_val_c = model_based_on_baseline(fit, val_c)

print("=== SO SÁNH ĐIỂM VALIDATION RMSLE (CÀNG THẤP CÀNG TỐT) ===")
print(f"1. Baseline gốc (weekday_median) trên Val gốc        : {rmsle(val.revenue.values, pred_baseline_val):.4f}")
print(f"2. Model phát triển từ Baseline trên Val gốc         : {rmsle(val.revenue.values, pred_model_val):.4f}  (Cải thiện -{rmsle(val.revenue.values, pred_baseline_val) - rmsle(val.revenue.values, pred_model_val):.4f})")
print(f"3. Model phát triển từ Baseline trên Val đã làm sạch : {rmsle(val_c.revenue.values, pred_model_val_c):.4f}")

# Kiểm định thêm trên Q1/2025 (cùng quý với tập Test Q1/2026)
fit_q1 = train[train.date < "2025-01-01"]
val_q1_c = train_c[(train_c.date >= "2025-01-01") & (train_c.date <= "2025-03-31")]
print(f"4. Baseline gốc (weekday_median) trên Q1/2025 Clean  : {rmsle(val_q1_c.revenue.values, weekday_median(fit_q1, val_q1_c)):.4f}")
print(f"5. Model phát triển từ Baseline trên Q1/2025 Clean   : {rmsle(val_q1_c.revenue.values, model_based_on_baseline(fit_q1, val_q1_c)):.4f}")

=== SO SÁNH ĐIỂM VALIDATION RMSLE (CÀNG THẤP CÀNG TỐT) ===
1. Baseline gốc (weekday_median) trên Val gốc        : 0.2387
2. Model phát triển từ Baseline trên Val gốc         : 0.1343  (Cải thiện -0.1044)
3. Model phát triển từ Baseline trên Val đã làm sạch : 0.1117
4. Baseline gốc (weekday_median) trên Q1/2025 Clean  : 0.2758


5. Model phát triển từ Baseline trên Q1/2025 Clean   : 0.1045


---
## 5. Huấn luyện trên toàn bộ `train` & Xuất file nộp bài (`submission.csv` và `submission_baseline.csv`)

In [5]:
# 1. Lưu mô hình Baseline gốc và xuất submission_baseline.csv
baseline_model = BaselineWeekdayModel().fit(train)
joblib.dump(baseline_model, "baseline_model.pkl")
sub_base = pd.DataFrame({"id": test.id, "revenue": np.round(baseline_model.predict(test), 3)})
sub_base.to_csv("submission_baseline.csv", index=False)

# 2. Huấn luyện & lưu Model nâng cấp từ Baseline trên 100% dữ liệu train và xuất submission.csv
enhanced_model = BaselineEnhancedCafeModel(stores, weather, holidays).fit(train)
joblib.dump(enhanced_model, "cafe_revenue_model.pkl")
sub = pd.DataFrame({"id": test.id, "revenue": np.round(enhanced_model.predict(test), 3)})
sub.to_csv("submission.csv", index=False)

print("Đã lưu 2 model: baseline_model.pkl & cafe_revenue_model.pkl")
print("Đã xuất 2 file nộp bài: submission_baseline.csv & submission.csv (3,869 dòng)")
print("\nSo sánh 5 dòng đầu giữa Baseline và Model nâng cấp từ Baseline:")
comp = pd.DataFrame({
    "id": test.id.head(5),
    "date": test.date.head(5).dt.date,
    "store_id": test.store_id.head(5),
    "baseline_revenue": sub_base.revenue.head(5),
    "model_revenue": sub.revenue.head(5)
})
comp

Đã lưu 2 model: baseline_model.pkl & cafe_revenue_model.pkl
Đã xuất 2 file nộp bài: submission_baseline.csv & submission.csv (3,869 dòng)

So sánh 5 dòng đầu giữa Baseline và Model nâng cấp từ Baseline:


,id,date,store_id,baseline_revenue,model_revenue
0,100000,2026-01-01,S01,11.880,6.529
1,100001,2026-01-01,S03,18.175,8.752
2,100002,2026-01-01,S04,17.120,21.394
3,100003,2026-01-01,S05,28.770,33.079
4,100004,2026-01-01,S06,18.510,9.531
